# 2.5D unstructured LES solver on an A100: cylinder Re 3900 (V3)

The unstructured collocated finite-volume solver in the x-y plane, Fourier in the periodic span, RK3 with a projection per stage, WALE, the whole step on CuPy (`src/upiso25.py`, `device="gpu"`). This notebook: (1) checks the environment, (2) profiles one step on this GPU on the actual mesh and prints the run-time estimate, (3) runs the correctness check, (4) launches the Re 3900 cylinder LES, (5) compares it with the published Re 3900 data (Parnaudeau et al. 2008 PIV, Kravchenko & Moin 2000, Lehmkuhl et al. 2013, Norberg).

The first cell finds the repository root above the notebook's folder (or clones the `unstructured` branch if the notebook was downloaded alone) and makes it the working directory; every path below is relative to it. The first code cell installs what is missing into this kernel (pyamg, cupy-cuda12x, ...). The shell cells (`!python ...`) use the same interpreter as the kernel.


In [ ]:
# 0a. Find the repository root (the notebook lives in tools/a100/, and Jupyter's working directory is
#     usually the notebook's own folder), or clone the repo if the notebook was downloaded on its own.
import os, subprocess
def find_root():
    d = os.getcwd()
    for _ in range(4):
        if os.path.exists(os.path.join(d, "src", "upiso25.py")): return d
        d = os.path.dirname(d)
    return None
root = find_root()
if root is None:
    print("repository not found above this directory; cloning the unstructured branch ...")
    subprocess.check_call(["git", "clone", "-q", "-b", "unstructured", "--depth", "1", "https://github.com/chandc/PICT-Python.git"]); root = os.path.abspath("PICT-Python")
os.chdir(root); print("working directory:", root)
# which version of the code this is: commit, branch, date, and whether the tree is clean. Compare with
# `git log -1 origin/unstructured` on GitHub; if behind, run  git pull  (or re-clone) and restart the kernel.
def git(*args):
    try: return subprocess.check_output(["git", *args], stderr=subprocess.DEVNULL).decode().strip()
    except Exception as e: return f"(git unavailable: {e})"
print("commit  :", git("log", "-1", "--format=%h %cd %s", "--date=short"))
print("branch  :", git("rev-parse", "--abbrev-ref", "HEAD"), "  dirty files:", len(git("status", "--porcelain").splitlines()))
remote = git("ls-remote", "https://github.com/chandc/PICT-Python.git", "refs/heads/unstructured").split()[:1]
head = git("rev-parse", "HEAD")
if remote and not head.startswith(remote[0]):
    # behind (or on another branch): bring this checkout to the tip of origin/unstructured
    print("checkout", head[:7], "is not origin/unstructured", remote[0][:7], "-> updating ...")
    print(git("fetch", "-q", "origin", "unstructured")); print(git("checkout", "-q", "-B", "unstructured", "FETCH_HEAD"))
    head = git("rev-parse", "HEAD"); print("now at  :", git("log", "-1", "--format=%h %cd %s", "--date=short"))
print("GitHub  :", remote[0][:7] if remote else "(offline)", "on origin/unstructured", "" if (remote and head.startswith(remote[0])) else "  <-- STILL different: delete the directory and re-run this cell to clone afresh")
assert os.path.exists("run_ucylinder3900.py") and os.path.exists("meshes/cylinder_re3900.msh"), "run_ucylinder3900.py / meshes/cylinder_re3900.msh missing: this is an old tree; delete it and re-run this cell"
print("driver  : run_ucylinder3900.py and meshes/cylinder_re3900.msh present  (ok)")
import importlib, sys as _s
for mod in [m for m in list(_s.modules) if m == "src" or m.startswith("src.")]: del _s.modules[mod]      # drop any solver modules a previous run imported from the old tree


## 0b. Google Drive: every output written there directly
Hosted runtimes lose their disk when the session ends. This mounts Drive (Colab) and sets `DRIVE_OUT = MyDrive/PICT-Python_runs/<tag>/`; the run cell passes it to the solver as `--outdir`, so the checkpoint, the final field, the statistics and the log are written to Drive as they are produced, not copied afterwards. `restore_from_drive()` and `sync_to_drive()` remain for figures and for older local files.

In [ ]:
import os, shutil, glob, threading, time
tag = "ucyl3900_cylinder_re3900_nz64_wale"
DRIVE_ROOT = None
try:
    from google.colab import drive                       # Colab
    drive.mount("/content/drive", force_remount=False); DRIVE_ROOT = "/content/drive/MyDrive/PICT-Python_runs"
except Exception as e:
    for cand in (os.path.expanduser("~/Google Drive/My Drive"), "/content/drive/MyDrive"):
        if os.path.isdir(cand): DRIVE_ROOT = os.path.join(cand, "PICT-Python_runs"); break
    if DRIVE_ROOT is None: print("no Google Drive here (not Colab, no local Drive folder):", type(e).__name__, "- outputs stay local")
if DRIVE_ROOT:
    DRIVE_OUT = os.path.join(DRIVE_ROOT, tag); os.makedirs(DRIVE_OUT, exist_ok=True); os.makedirs(os.path.join(DRIVE_OUT, "logs"), exist_ok=True); print("Drive folder:", DRIVE_OUT)

def sync_to_drive(verbose=True):
    """copy this run's outputs to Drive if newer than the copy there; safe to call any time"""
    if not DRIVE_ROOT: return
    n = 0
    for pat, sub in ((f"results/{tag}*.npz", ""), (f"results/logs/{tag}*.log", "logs"), ("figures/*re3900*.png", ""), ("figures/*3900*.png", "")):
        for f in glob.glob(pat):
            dst = os.path.join(DRIVE_OUT, sub, os.path.basename(f))
            if not os.path.exists(dst) or os.path.getmtime(f) > os.path.getmtime(dst) + 1:
                shutil.copy2(f, dst); n += 1
    if verbose: print(time.strftime("%H:%M:%S"), f"synced {n} file(s) to Drive")

def _sync_loop(period=600):
    while True:
        time.sleep(period)
        try: sync_to_drive(verbose=False)
        except Exception as e: print("drive sync failed:", e)

def restore_from_drive():
    """bring a checkpoint/log back from Drive into results/ (new session, or after the local disk was lost)"""
    if not DRIVE_ROOT: return None
    os.makedirs("results/logs", exist_ok=True); got = []
    for f in glob.glob(os.path.join(DRIVE_OUT, f"{tag}*.npz")) + glob.glob(os.path.join(DRIVE_OUT, "logs", f"{tag}*.log")):
        dst = os.path.join("results", "logs" if f.endswith(".log") else "", os.path.basename(f))
        if not os.path.exists(dst): shutil.copy2(f, dst); got.append(dst)
    print("restored from Drive:", got or "nothing new"); return got
restore_from_drive()

In [ ]:
# 0. Dependencies into THIS kernel (pip of the running interpreter, not the shell's). pyamg builds the
#    multigrid hierarchy on the host; cupy-cuda12x is the GPU array library (binary wheels on x86-64).
import sys, subprocess, importlib
def ensure(mod, pkg):
    try: importlib.import_module(mod); print(f"ok  {mod}")
    except ImportError:
        print(f"installing {pkg} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg]); importlib.invalidate_caches(); importlib.import_module(mod); print(f"ok  {mod} (installed)")
ensure("numpy", "numpy>=1.26"); ensure("scipy", "scipy>=1.11"); ensure("pyamg", "pyamg>=5.0"); ensure("matplotlib", "matplotlib>=3.7")
# CuPy must match the DRIVER's CUDA version, not just be importable: a wheel built for CUDA 13 on a machine
# whose driver supports 12.x imports fine and then fails on the first array with
# "cudaErrorInsufficientDriver". Test a real device operation and reinstall the right wheel if needed.
import re
def cupy_works():
    try:
        import cupy; cupy.zeros(1).sum(); print("ok  cupy", cupy.__version__, "on", cupy.cuda.runtime.getDeviceProperties(0)["name"]); return True
    except Exception as e:
        print("cupy not usable:", type(e).__name__, str(e)[:160]); return False
if not cupy_works():
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout; m_ = re.search(r"CUDA Version: (\d+)\.(\d+)", smi)
    major = int(m_.group(1)) if m_ else 12; pkg = "cupy-cuda12x" if major < 13 else "cupy-cuda13x"
    print(f"driver supports CUDA {m_.group(0) if m_ else '?'} -> installing {pkg} (removing any other cupy build)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "cupy", "cupy-cuda12x", "cupy-cuda13x", "cupy-cuda11x"], capture_output=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    raise SystemExit(f"{pkg} installed. RESTART THE KERNEL/RUNTIME (Runtime > Restart session), then run the cells again from the top.")


In [ ]:
import os, sys, subprocess, time; sys.path.insert(0, os.getcwd())      # cell 0a set the working directory to the repo root
import numpy as np, cupy as cp
p = cp.cuda.runtime.getDeviceProperties(0); print(p["name"], f"{p['totalGlobalMem']/2**30:.0f} GB, {p['multiProcessorCount']} SMs, cupy {cp.__version__}")
import pyamg, scipy; print("pyamg", pyamg.__version__, "scipy", scipy.__version__, "numpy", np.__version__)   # if this fails, rerun cell 0

## 1. Profile one step on this GPU, on the actual mesh
`meshes/cylinder_re3900.msh`: 60,000 quads (wall cell 0.003 D, 0.04 D cells to x = 6 D, |y| < 10 D, outlet 20 D) × 64 Fourier planes over L_z = πD = 3.84 M cell-modes, WALE, three non-orthogonal pressure passes. The channel Re_τ 395 run measured 20 ms per 10⁵ cell-modes on an A100 (380–440 ms per step at 1.97 M cell-modes); this mesh's step should come out near 1.0–1.4 s. The run is ~75,000 steps (150 D/U at dt ≈ 0.002), i.e. 20–30 h — plan on two or three Colab sessions; the run cell resumes from the Drive checkpoint by itself.


In [ ]:
!{sys.executable} tools/a100/profile_step.py --mesh meshes/cylinder_re3900.msh --nz 64 --steps 5


## 2. Correctness on this device (2 min)
The 3D Taylor–Green energy balance: −dE/dt must equal the scheme's own discrete dissipation to 0.1–0.2% at every sample (record section 51), E₀ = 31.006277.

In [ ]:
!TGV_DEVICE=gpu TGV_SOLVER=amg TGV_RE=100 TGV_T=2 TGV_N=32 TGV_NZ=32 TGV_DT=0.02 {sys.executable} test_utgv3d.py A

## 3. The cylinder at Re 3900

Setup (`run_ucylinder3900.py`): D = U = 1, ν = 1/3900; inlet u = 1 at x = −10 D, slip free-stream at |y| = 10 D (blockage 5%), p = 0 outlet at x = 20 D, no-slip cylinder; span L_z = πD with 64 Fourier planes (Δz 0.049 D); WALE; impulsive start with a y-odd v perturbation and a spanwise-periodic w perturbation in the near wake. `--cfl-max 0.8` on dt₀ 0.004 (face-flux Courant number; the step halves as the shoulder velocity builds — expect dt 0.002). **The SGS term is switched on at t = 2** (`--t-sgs`): at t = 0 the impulsive start puts u = 1 in the 0.003 D wall cells and WALE turns that gradient into ν_t ≈ 5000 ν, which diverges in two steps; two time units later the boundary layer exists and the model is well behaved.

Statistics from t = 50 to 150 (about 21 shedding periods at St 0.21): span-and-time means of u, v, p, u'u', v'v', u'v', w'w', ν_t per cell and the time-mean wall pressure per wall face, written with every checkpoint (`<tag>_stats_partial.npz`) and at the end (`<tag>_stats.npz`); forces every step in `<tag>_hist.npy`. Every output file goes straight to the Drive folder (`--outdir`); if the session drops, run the cell again in a new session and it continues from the checkpoint there.

Reference values, all at Re 3900 with a πD span: St 0.208 (Parnaudeau PIV) / 0.21; C_D 0.98 (Norberg) – 1.04 (Kravchenko & Moin); C_pb −0.88 (Norberg) / −0.94 (K&M); recirculation length L_r 1.51 D (Parnaudeau PIV), 1.35 (K&M), 1.26 or 1.55 (Lehmkuhl DNS, two states). Criteria (LES plan V3): St within 3%, C_D within 5%, L_r within 10% of the Parnaudeau band.


In [ ]:
# The run, in the FOREGROUND of this cell (a running cell keeps the session alive; a nohup'd process does not),
# writing every output straight to Drive (--outdir), and resuming automatically from the checkpoint there.
# If the session still drops, just run this cell again in the new session: it continues where the checkpoint is.
assert os.path.exists("run_ucylinder3900.py"), "old tree: re-run cell 0a"
OUT = DRIVE_OUT if DRIVE_ROOT else "results"; os.makedirs(os.path.join(OUT, "logs"), exist_ok=True)
ck = os.path.join(OUT, f"{tag}_ckpt.npz"); log_path = os.path.join(OUT, "logs", f"{tag}.log")
cmd = [sys.executable, "-u", "run_ucylinder3900.py", "--device", "gpu", "--mesh", "meshes/cylinder_re3900.msh", "--nz", "64", "--dt", "0.004", "--cfl-max", "0.8", "--sgs", "wale", "--t-sgs", "2",
       "--T", "150", "--t-stats", "50", "--report", "250", "--checkpoint", "2000", "--tag", tag, "--outdir", OUT] + (["--restart", ck] if os.path.exists(ck) else [])
print("resuming from", ck if os.path.exists(ck) else "scratch"); print(" ".join(cmd)); print("log:", log_path, flush=True)
with open(log_path, "a") as logf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:                       # stream: the cell shows progress live and the log on Drive grows line by line
        if "Warning" in line: continue
        print(line, end="", flush=True); logf.write(line); logf.flush()
    proc.wait(); print("exit code", proc.returncode)

## 4. Compare with the Re 3900 references
When the log shows `RESULT ...` (or from the interim `_stats_partial.npz` at any checkpoint after t = 50): force history and Strouhal number, C_p around the cylinder, the mean streamwise velocity on the wake centreline (recirculation length), and the cross-wake mean and rms profiles at x = 1.06, 1.54, 2.02 D (Parnaudeau's PIV stations).


In [ ]:
sf = os.path.join(OUT, f"{tag}_stats.npz"); sf = sf if os.path.exists(sf) else os.path.join(OUT, f"{tag}_stats_partial.npz")
print("statistics file:", sf)
!{sys.executable} plot_utility/plot_ucylinder_re3900.py "{sf}"
from IPython.display import Image, display
for f in ("figures/ucylinder_re3900_forces.png", "figures/ucylinder_re3900_wake.png", "figures/ucylinder_re3900_fields.png"):
    if os.path.exists(f): display(Image(f)); shutil.copy2(f, OUT)


## 5. Notes
* `--nz 48` and `--T 125 --t-stats 50` halve the cost (about 12 h) and still settle St and C_D; the recirculation length converges slowest and wants the full window.
* `--sgs none` is the implicit-model control; `--sgs smagorinsky` runs undamped here (van Driest needs a wall distance the cylinder driver does not compute).
* Wall cell 0.003 D at the shoulders, 0.005 D at the block corners (±45°); the laminar boundary layer at separation is 0.02–0.03 D thick.
* If the recirculation length comes out short of the Parnaudeau band with St and C_D on target, the WALE transition behaviour seen in the Taylor–Green case (record §58) is the first suspect and the σ-model port is the next step, not a finer mesh.
